# SparkSession

In [1]:
import os
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("S3Example") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", os.getenv("MINIO_ROOT_USER")) \
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD")) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .getOrCreate()

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
ru.yandex.clickhouse#clickhouse-jdbc added as a dependency
org.postgresql#postgresql added as a dependency
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-e0d68440-bab8-4b14-820d-fa00548e577a;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found ru.yandex.clickhouse#clickhouse-jdbc;0.3.2 in central
	found com.clickhouse#clickhouse-http-client;0.3.2 in central
	found com.clickhouse#clickhouse-client;0.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found com.google.code.gson#gson;2.8.8 in central
	found org.apache.httpcomponents#

In [2]:
spark.sparkContext.setLogLevel("WARN")

In [3]:
df_hugg = spark\
            .read\
            .parquet('source/subset_100_1.parquet', header=True, inferSchema=True)

In [12]:
df = df_hugg.groupBy('open_type').count().show()

+---------------+-----+
|      open_type|count|
+---------------+-----+
|       Open Web| 9624|
|   Open Science| 1912|
|  Semantic data| 3120|
|   Open Culture| 8119|
|    Open Source|20284|
|Open Government| 7849|
+---------------+-----+



In [20]:
crime_df = spark\
            .read\
            .csv('source/crimes-in-Boston/crime.csv', header=True, inferSchema=True)

offense_codes_df = spark\
            .read\
            .csv('source/crimes-in-Boston/offense_codes.csv', header=True, inferSchema=True)

In [21]:
crime_df.printSchema()

root
 |-- INCIDENT_NUMBER: string (nullable = true)
 |-- OFFENSE_CODE: integer (nullable = true)
 |-- OFFENSE_CODE_GROUP: string (nullable = true)
 |-- OFFENSE_DESCRIPTION: string (nullable = true)
 |-- DISTRICT: string (nullable = true)
 |-- REPORTING_AREA: string (nullable = true)
 |-- SHOOTING: string (nullable = true)
 |-- OCCURRED_ON_DATE: timestamp (nullable = true)
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: string (nullable = true)
 |-- HOUR: integer (nullable = true)
 |-- UCR_PART: string (nullable = true)
 |-- STREET: string (nullable = true)
 |-- Lat: double (nullable = true)
 |-- Long: double (nullable = true)
 |-- Location: string (nullable = true)



 # Select column and filtering   

In [22]:
column = ['INCIDENT_NUMBER', 'OFFENSE_CODE', 'OFFENSE_CODE_GROUP', 'OFFENSE_DESCRIPTION']
selected_crime_df = crime_df.select(column).where(crime_df['STREET'] == 'WASHINGTON ST')

In [23]:
selected_crime_df.explain()

== Physical Plan ==
*(1) Project [INCIDENT_NUMBER#254, OFFENSE_CODE#255, OFFENSE_CODE_GROUP#256, OFFENSE_DESCRIPTION#257]
+- *(1) Filter (isnotnull(STREET#267) AND (STREET#267 = WASHINGTON ST))
   +- FileScan csv [INCIDENT_NUMBER#254,OFFENSE_CODE#255,OFFENSE_CODE_GROUP#256,OFFENSE_DESCRIPTION#257,STREET#267] Batched: false, DataFilters: [isnotnull(STREET#267), (STREET#267 = WASHINGTON ST)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/source/crimes-in-Boston/crime.csv], PartitionFilters: [], PushedFilters: [IsNotNull(STREET), EqualTo(STREET,WASHINGTON ST)], ReadSchema: struct<INCIDENT_NUMBER:string,OFFENSE_CODE:int,OFFENSE_CODE_GROUP:string,OFFENSE_DESCRIPTION:stri...




In [24]:
offense_codes_df.printSchema()

root
 |-- CODE: integer (nullable = true)
 |-- NAME: string (nullable = true)



In [26]:
offense_codes_df.count()

576

In [33]:
df = crime_df.join(offense_codes_df, (crime_df.OFFENSE_CODE == offense_codes_df.CODE), 'inner')

In [31]:
df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [OFFENSE_CODE#255], [CODE#305], Inner, BuildRight, false
   :- Filter isnotnull(OFFENSE_CODE#255)
   :  +- FileScan csv [INCIDENT_NUMBER#254,OFFENSE_CODE#255,OFFENSE_CODE_GROUP#256,OFFENSE_DESCRIPTION#257,DISTRICT#258,REPORTING_AREA#259,SHOOTING#260,OCCURRED_ON_DATE#261,YEAR#262,MONTH#263,DAY_OF_WEEK#264,HOUR#265,UCR_PART#266,STREET#267,Lat#268,Long#269,Location#270] Batched: false, DataFilters: [isnotnull(OFFENSE_CODE#255)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/source/crimes-in-Boston/crime.csv], PartitionFilters: [], PushedFilters: [IsNotNull(OFFENSE_CODE)], ReadSchema: struct<INCIDENT_NUMBER:string,OFFENSE_CODE:int,OFFENSE_CODE_GROUP:string,OFFENSE_DESCRIPTION:stri...
   +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [plan_id=225]
      +- Filter isnotnull(CODE#305)
         +- FileScan csv [CODE#305,NAME#306] Batched: false

In [34]:
df.show()

+---------------+------------+--------------------+--------------------+--------+--------------+--------+-------------------+----+-----+-----------+----+----------+-----------------+-----------+------------+--------------------+----+--------------------+
|INCIDENT_NUMBER|OFFENSE_CODE|  OFFENSE_CODE_GROUP| OFFENSE_DESCRIPTION|DISTRICT|REPORTING_AREA|SHOOTING|   OCCURRED_ON_DATE|YEAR|MONTH|DAY_OF_WEEK|HOUR|  UCR_PART|           STREET|        Lat|        Long|            Location|CODE|                NAME|
+---------------+------------+--------------------+--------------------+--------+--------------+--------+-------------------+----+-----+-----------+----+----------+-----------------+-----------+------------+--------------------+----+--------------------+
|     I182070945|         619|             Larceny|  LARCENY ALL OTHERS|     D14|           808|    NULL|2018-09-02 13:00:00|2018|    9|     Sunday|  13|  Part One|       LINCOLN ST|42.35779134|-71.13937053|(42.35779134, -71...| 619|LA

# Чтение из Kafka

In [3]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, DataType
from pyspark.sql.functions import from_json, col
from pyspark.sql.functions import expr
from pyspark.sql import functions as F

kafka_topic = "khd.dev_cbrspb_tmd.etl_pkg"
kafka_bootstrap = "kafka:29093"


# Чтение из Kafka
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", kafka_bootstrap) \
    .option("subscribe", kafka_topic) \
    .option("startingOffsets", "latest") \
    .load()


# Описание схемы JSON сообщения
schema = StructType([
    StructField("before", StructType([
        StructField("pkg_sqn", IntegerType(), True),
        StructField('data_domain_id', StringType(), True),
        StructField("pkg_nm", StringType(), True),
        StructField("change_dttm", IntegerType(), True)
    ]), True),
    StructField("after", StructType([
        StructField("pkg_sqn", IntegerType(), True),
        StructField('data_domain_id', StringType(), True),
        StructField("pkg_nm", StringType(), True),
        StructField("change_dttm", IntegerType(), True)
    ]), True),
    StructField("source", StructType([]), True),  # если не используешь, можно пустым
    StructField("op", StringType(), True),
    StructField("ts_ms", LongType(), True)
])

# Распарсенные данные
json_df = df.selectExpr("CAST(value AS STRING) as json_str") \
    .select(from_json("json_str", schema).alias("data")) \
    .where("data.after IS NOT NULL") \
    .select("data.after.*") \
    .withColumn('change_dttm2', expr("date_add('1970-01-01', change_dttm)"))


In [17]:
# Вывод в консоль
json_df.writeStream \
    .format("console") \
    .option("truncate", False) \
    .start() \
    .awaitTermination()

25/06/28 08:05:26 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-75f43220-8d9e-4f29-a0f2-d06204e69598. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
25/06/28 08:05:26 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
25/06/28 08:05:26 WARN AdminClientConfig: The configuration 'key.deserializer' was supplied but isn't a known config.
25/06/28 08:05:26 WARN AdminClientConfig: The configuration 'value.deserializer' was supplied but isn't a known config.
25/06/28 08:05:26 WARN AdminClientConfig: The configuration 'enable.auto.commit' was supplied but isn't a known config.
25/06/28 08:05:26 WARN AdminClientConfig: The configuration 'max.poll.records' was supplied but isn't a known con

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+--------------+------+-----------+------------+
|pkg_sqn|data_domain_id|pkg_nm|change_dttm|change_dttm2|
+-------+--------------+------+-----------+------------+
+-------+--------------+------+-----------+------------+

-------------------------------------------
Batch: 24
-------------------------------------------
-------------------------------------------
Batch: 9
-------------------------------------------
+-------+---------------------------------------------+---------+-----------+
|pkg_sqn|data_domain_id                               |pkg_nm   |change_dttm|
+-------+---------------------------------------------+---------+-----------+
|441    |http://www.it.ru/Schemas/Avior/some_schema441|pkg_nm441|20267      |
|442    |http://www.it.ru/Schemas/Avior/some_schema442|pkg_nm442|20267      |
|443    |http://www.it.ru/Schemas/Avior/some_schema443|pkg_nm443|20267      |
|444    |ht

-------------------------------------------
Batch: 26
-------------------------------------------
+-------+---------------------------------------------+---------+-----------+
|pkg_sqn|data_domain_id                               |pkg_nm   |change_dttm|
+-------+---------------------------------------------+---------+-----------+
|451    |http://www.it.ru/Schemas/Avior/some_schema451|pkg_nm451|20267      |
|452    |http://www.it.ru/Schemas/Avior/some_schema452|pkg_nm452|20267      |
|453    |http://www.it.ru/Schemas/Avior/some_schema453|pkg_nm453|20267      |
|454    |http://www.it.ru/Schemas/Avior/some_schema454|pkg_nm454|20267      |
|455    |http://www.it.ru/Schemas/Avior/some_schema455|pkg_nm455|20267      |
|456    |http://www.it.ru/Schemas/Avior/some_schema456|pkg_nm456|20267      |
|457    |http://www.it.ru/Schemas/Avior/some_schema457|pkg_nm457|20267      |
+-------+---------------------------------------------+---------+-----------+



-------------------------------------------
Batch: 11
-------------------------------------------
-------------------------------------------
Batch: 39
-------------------------------------------


-------------------------------------------
Batch: 3
-------------------------------------------
+-------+---------------------------------------------+---------+-----------+
|pkg_sqn|data_domain_id                               |pkg_nm   |change_dttm|
+-------+---------------------------------------------+---------+-----------+
|451    |http://www.it.ru/Schemas/Avior/some_schema451|pkg_nm451|20267      |
|452    |http://www.it.ru/Schemas/Avior/some_schema452|pkg_nm452|20267      |
|453    |http://www.it.ru/Schemas/Avior/some_schema453|pkg_nm453|20267      |
|454    |http://www.it.ru/Schemas/Avior/some_schema454|pkg_nm454|20267      |
|455    |http://www.it.ru/Schemas/Avior/some_schema455|pkg_nm455|20267      |
|456    |http://www.it.ru/Schemas/Avior/some_schema456|pkg_nm456|20267      |
|457    |http://www.it.ru/Schemas/Avior/some_schema457|pkg_nm457|20267      |
+-------+---------------------------------------------+---------+-----------+

+-------+-----------------------------------

-------------------------------------------
Batch: 41
-------------------------------------------
+-------+---------------------------------------------+---------+-----------+
|pkg_sqn|data_domain_id                               |pkg_nm   |change_dttm|
+-------+---------------------------------------------+---------+-----------+
|461    |http://www.it.ru/Schemas/Avior/some_schema461|pkg_nm461|20267      |
|462    |http://www.it.ru/Schemas/Avior/some_schema462|pkg_nm462|20267      |
|463    |http://www.it.ru/Schemas/Avior/some_schema463|pkg_nm463|20267      |
|464    |http://www.it.ru/Schemas/Avior/some_schema464|pkg_nm464|20267      |
|465    |http://www.it.ru/Schemas/Avior/some_schema465|pkg_nm465|20267      |
|466    |http://www.it.ru/Schemas/Avior/some_schema466|pkg_nm466|20267      |
|467    |http://www.it.ru/Schemas/Avior/some_schema467|pkg_nm467|20267      |
|468    |http://www.it.ru/Schemas/Avior/some_schema468|pkg_nm468|20267      |
|469    |http://www.it.ru/Schemas/Avior/some

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 706, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [4]:
# Преобразуем ts (Unix → UTC Timestamp → Date)
# processed_df = json_df \
#     .withColumn("ts_sec", (col("ts") / 1_000_000).cast("double")) \
#     .withColumn("ts_utc", from_unixtime(col("ts_sec")).cast("timestamp")) \
#     .withColumn("event_date", to_date(col("ts_utc"))) \
#     .drop("ts", "ts_sec")

args = {'s3_path': 's3a://prod/stream/etl_pkg/'}

hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

In [5]:
# Запись в S3 с партиционированием
json_df.writeStream \
    .format("parquet") \
    .queryName("etl_pkg") \
    .option("path", args['s3_path']) \
    .option("checkpointLocation", args['s3_path'] + "/_checkpoint/") \
    .partitionBy("change_dttm2") \
    .outputMode("append") \
    .start() \
    .awaitTermination()


25/06/28 08:40:56 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
25/06/28 08:40:59 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
25/06/28 08:41:12 WARN AdminClientConfig: The configuration 'key.deserializer' was supplied but isn't a known config.
25/06/28 08:41:12 WARN AdminClientConfig: The configuration 'value.deserializer' was supplied but isn't a known config.
25/06/28 08:41:12 WARN AdminClientConfig: The configuration 'enable.auto.commit' was supplied but isn't a known config.
25/06/28 08:41:12 WARN AdminClientConfig: The configuration 'max.poll.records' was supplied but isn't a known config.
25/06/28 08:41:12 WARN AdminClientConfig: The configuration 'auto.offset.reset' was supplied but isn't a known config.
ERROR:root:KeyboardInterrupt while sending command.                             
Traceback (most recent call last):
  

KeyboardInterrupt: 